# Null Catamenial Epilepsy Analysis Notebook

Populated from `outputs` outputs. Detected analysis mode: **full**.

- Participants: **100,000** (healthy ovulatory: 50000, heterogeneous menstruating-age: 50000)
- Primary window rows: **900,000**
- Study-level Monte Carlo rows: **340,000**
- Manifest files: **65**

## Cohort terminology

This notebook uses **heterogeneous menstruating-age** as the presentation label for the broader cohort key stored in the analysis files. In this null-simulation study it means an assumption-driven broader menstruating-age simulated cohort, not a disease-positive, clinically diagnosed, or demographically representative population. This cohort allows the hormone-cycle simulator's natural ovulatory and anovulatory behavior and its configured rates of cycle modifiers such as PCOS, peri-menarche, perimenopause, dysmenorrhea, and cycle irregularity when available. It is contrasted with the **healthy ovulatory** cohort, which is restricted to adult ovulatory cycling with those medical modifiers disabled where the simulator exposes those controls. In both cohorts, seizure diaries are generated independently from menstrual diaries and then circularly shifted before merging, so any apparent catamenial epilepsy classification is a false positive under the null. Positive-control true-coupling modes are generated into separate output directories and should be interpreted as operating-characteristic sensitivities.

## Exact analysis plan followed

1. Simulate two cohorts separately and never pool results: healthy ovulatory and heterogeneous menstruating-age.
2. Full defined cohort sizes are `{'healthy ovulatory': 50000, 'heterogeneous menstruating-age': 50000}`; smoke mode uses `100` total participants.
3. For each participant, simulate an independent CHOCOLATES seizure diary and an independent hormone-cycle diary for `36` months in full mode.
4. Apply a random circular shift before merging to break hidden start-date alignment.
5. Label Herzog phases on the full diary before subsetting windows.
6. Sample calendar windows, full 36-month windows, and complete-cycle windows exactly as configured.
7. Classify windows using exact Herzog 2004, windowed Herzog thresholds, C3-exclusion and pattern-only sensitivities, minimum-data rules, reproducibility rules, stabilized/window-dispersion NB regression, and assumption-based historical definitions.
8. Summarize false positives and indeterminacy by cohort, window, definition, seizure-frequency stratum, cycle-regularity stratum, trial-like subsets, and study-level Monte Carlo benchmarks.
9. Save outputs as parquet/CSV, publication figures as PNG/PDF, a 1% daily audit sample, and a manifest.

### Recorded assumptions

- Definition D uses a participant-full-diary method-of-moments negative-binomial alpha recorded in d_alpha; Poisson robust fallback is recorded in d_reason when statsmodels NB fitting fails. Definition D_window_alpha re-estimates alpha from the analyzed window as a non-oracle sensitivity.
- Healthy ovulatory cohort used hormone_cycler build_patient_profile/render_cycle with ovulation_probability set to 1.0 because simulate_diary does not expose a public force-ovulation knob.
- Historical definitions H1-H4 are assumption-based operationalizations and are flagged in summary outputs.
- Study-level Monte Carlo samples each selected participant from a deterministic pool of precomputed random valid 3-month windows to avoid retaining all daily diaries in memory.
- The hormone simulator exposes medical-factor knobs but no natural prevalence sampler; heterogeneous menstruating-age medical factors were sampled from config.yaml rates.

## Reproducible function calls

These cells are the exact calls used to regenerate the analysis outputs. Run the smoke call for a quick end-to-end check; run the full call for the defined 100,000-participant analysis.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from paper1_null_ce.core.utils import load_config
from paper1_null_ce.core.simulate import run_pipeline

config = load_config(ROOT / "config.yaml")

# Quick validation run used while developing and reviewing the pipeline:
smoke_result = run_pipeline(config, mode="smoke")

# Prespecified full analysis. This is intentionally separate because it is large:
# full_result = run_pipeline(config, mode="full")

# During a long run, check ETA from another terminal:
# python3.11 scripts/check_paper1_progress.py --progress outputs/progress.json


## Load the current populated outputs

The remaining notebook cells read the existing output artifacts. This keeps figure and table rendering fast and reproducible after either a smoke run or a full run.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = ROOT / "outputs"

participant_summary = pd.read_parquet(OUTPUT_DIR / "participant_summary.parquet")
window_results = pd.read_parquet(OUTPUT_DIR / "window_results.parquet")
study_level = pd.read_parquet(OUTPUT_DIR / "study_level_3month_n30.parquet")
summary_tables = pd.read_csv(OUTPUT_DIR / "summary_tables.csv")
manifest = json.loads((OUTPUT_DIR / "manifest.json").read_text())

participant_summary.shape, window_results.shape, study_level.shape, summary_tables.shape


## Table 1. Cohort and diary summary

**Why this table is included.** This table verifies that both defined cohorts are represented separately, that cycle summaries are available, and that seizure-burden metrics were carried through from the seizure simulator. It is the first QC table because every downstream apparent-classification estimate depends on the cohort construction and diary burden.

**Code to call.**

In [ ]:
cohort_summary = (
    participant_summary
    .groupby("cohort")
    .agg(
        participants=("participant_id", "nunique"),
        age_mean=("age", "mean"),
        age_sd=("age", "std"),
        mean_cycle_length=("mean_cycle_length", "mean"),
        sd_cycle_length=("sd_cycle_length", "mean"),
        ovulatory_fraction=("ovulatory_fraction", "mean"),
        seizure_days_per_month=("seizure_days_per_month", "mean"),
        seizures_per_month=("seizures_per_month", "mean"),
    )
    .reset_index()
)
cohort_summary


| Cohort                         | Participants | Mean age, years | Age SD, years | Mean cycle length, days | Mean cycle-length SD, days | Ovulatory cycles | Seizure days per month | Seizures per month |
| ------------------------------ | ------------ | --------------- | ------------- | ----------------------- | -------------------------- | ---------------- | ---------------------- | ------------------ |
| healthy ovulatory              | 50,000       | 31.5            | 7.8           | 29.15                   | 3.39                       | 100.0%           | 2.46                   | 6.83               |
| heterogeneous menstruating-age | 50,000       | 34.0            | 12.1          | 30.92                   | 4.74                       | 79.0%            | 2.45                   | 6.80               |

**Table 1 caption.** Cohort-level participant and diary summaries for the full simulation. Percentages use a 0-100% scale; seizure rates are monthly averages over the 36-month diary.

## Table 2. Primary full-window false-positive rates

**Why this table is included.** This table is the primary result summary for person-window false-positive rates under the null. It uses the full diary window and reports classifiable denominators, positives, Wilson 95% intervals, and indeterminate rates separately by cohort and definition.

**Code to call.**

In [ ]:
primary_full = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.subset == "all")
    & (summary_tables.window_type == "full")
    & (summary_tables.definition.isin([
        "A_windowed_any", "B_minimum_data_any",
        "C_reproducibility_any", "D_nb_regression_any"
    ]))
].copy()
primary_full


| Cohort                         | CE definition                                       | Windows analyzed | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows |
| ------------------------------ | --------------------------------------------------- | ---------------- | -------------------- | ---------------------- | ---------------------------- | --------------------- |
| healthy ovulatory              | Windowed Herzog thresholds                          | 50,000           | 49,605               | 5,576                  | 11.2% (11.0, 11.5)           | 0.8%                  |
| healthy ovulatory              | Windowed Herzog with minimum data                   | 50,000           | 48,359               | 4,940                  | 10.2% (9.9, 10.5)            | 3.3%                  |
| healthy ovulatory              | Cycle reproducibility, 6-cycle rule                 | 50,000           | 6,132                | 1                      | 0.0% (0.0, 0.1)              | 87.7%                 |
| healthy ovulatory              | Negative-binomial regression, stabilized dispersion | 50,000           | 48,359               | 1,966                  | 4.1% (3.9, 4.2)              | 3.3%                  |
| heterogeneous menstruating-age | Windowed Herzog thresholds                          | 50,000           | 49,587               | 17,983                 | 36.3% (35.8, 36.7)           | 0.8%                  |
| heterogeneous menstruating-age | Windowed Herzog with minimum data                   | 50,000           | 48,333               | 17,206                 | 35.6% (35.2, 36.0)           | 3.3%                  |
| heterogeneous menstruating-age | Cycle reproducibility, 6-cycle rule                 | 50,000           | 13,364               | 1,605                  | 12.0% (11.5, 12.6)           | 73.3%                 |
| heterogeneous menstruating-age | Negative-binomial regression, stabilized dispersion | 50,000           | 48,333               | 2,032                  | 4.2% (4.0, 4.4)              | 3.3%                  |

**Table 2 caption.** Primary full-diary false-positive rates under the null. The denominator for the false-positive rate is the number of classifiable participant windows, and the confidence interval is Wilson 95%.

## Table 3. Window-length sensitivity for core definitions

**Why this table is included.** This table shows why diary length matters. Short calendar windows can be classifiable for simple windowed ratios but not for minimum-data, reproducibility, or exact three-cycle rules; the indeterminate column quantifies that tradeoff.

**Code to call.**

In [ ]:
window_sensitivity = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.subset == "all")
    & (summary_tables.definition.isin([
        "A_exact_any", "A_windowed_any", "B_minimum_data_any",
        "C_reproducibility_any", "D_nb_regression_any"
    ]))
].copy()
window_sensitivity


| Cohort                         | Observation window  | CE definition                                       | Classifiable windows | False-positive windows | False-positive rate | Indeterminate windows |
| ------------------------------ | ------------------- | --------------------------------------------------- | -------------------- | ---------------------- | ------------------- | --------------------- |
| healthy ovulatory              | 1 month             | Exact Herzog 2004, any CE pattern                   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 1 month             | Windowed Herzog thresholds                          | 38,234               | 18,249                 | 47.7%               | 23.5%                 |
| healthy ovulatory              | 1 month             | Windowed Herzog with minimum data                   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 1 month             | Cycle reproducibility, 6-cycle rule                 | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 1 month             | Negative-binomial regression, stabilized dispersion | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 12 cycles           | Exact Herzog 2004, any CE pattern                   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 12 cycles           | Windowed Herzog thresholds                          | 48,682               | 12,364                 | 25.4%               | 2.6%                  |
| healthy ovulatory              | 12 cycles           | Windowed Herzog with minimum data                   | 45,088               | 10,544                 | 23.4%               | 9.8%                  |
| healthy ovulatory              | 12 cycles           | Cycle reproducibility, 6-cycle rule                 | 15,706               | 355                    | 2.3%                | 68.6%                 |
| healthy ovulatory              | 12 cycles           | Negative-binomial regression, stabilized dispersion | 45,088               | 2,155                  | 4.8%                | 9.8%                  |
| healthy ovulatory              | 12 months           | Exact Herzog 2004, any CE pattern                   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 12 months           | Windowed Herzog thresholds                          | 48,775               | 12,419                 | 25.5%               | 2.4%                  |
| healthy ovulatory              | 12 months           | Windowed Herzog with minimum data                   | 45,266               | 10,661                 | 23.6%               | 9.5%                  |
| healthy ovulatory              | 12 months           | Cycle reproducibility, 6-cycle rule                 | 15,964               | 246                    | 1.5%                | 68.1%                 |
| healthy ovulatory              | 12 months           | Negative-binomial regression, stabilized dispersion | 45,266               | 2,207                  | 4.9%                | 9.5%                  |
| healthy ovulatory              | 3 cycles            | Exact Herzog 2004, any CE pattern                   | 23,101               | 11,616                 | 50.3%               | 53.8%                 |
| healthy ovulatory              | 3 cycles            | Windowed Herzog thresholds                          | 45,122               | 18,831                 | 41.7%               | 9.8%                  |
| healthy ovulatory              | 3 cycles            | Windowed Herzog with minimum data                   | 17                   | 12                     | 70.6%               | 100.0%                |
| healthy ovulatory              | 3 cycles            | Cycle reproducibility, 6-cycle rule                 | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 cycles            | Negative-binomial regression, stabilized dispersion | 17                   | 1                      | 5.9%                | 100.0%                |
| healthy ovulatory              | 3 months            | Exact Herzog 2004, any CE pattern                   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 months            | Windowed Herzog thresholds                          | 45,280               | 18,696                 | 41.3%               | 9.4%                  |
| healthy ovulatory              | 3 months            | Windowed Herzog with minimum data                   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 months            | Cycle reproducibility, 6-cycle rule                 | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 months            | Negative-binomial regression, stabilized dispersion | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 36-month full diary | Exact Herzog 2004, any CE pattern                   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 36-month full diary | Windowed Herzog thresholds                          | 49,605               | 5,576                  | 11.2%               | 0.8%                  |
| healthy ovulatory              | 36-month full diary | Windowed Herzog with minimum data                   | 48,359               | 4,940                  | 10.2%               | 3.3%                  |
| healthy ovulatory              | 36-month full diary | Cycle reproducibility, 6-cycle rule                 | 6,132                | 1                      | 0.0%                | 87.7%                 |
| healthy ovulatory              | 36-month full diary | Negative-binomial regression, stabilized dispersion | 48,359               | 1,966                  | 4.1%                | 3.3%                  |
| healthy ovulatory              | 4 months            | Exact Herzog 2004, any CE pattern                   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 4 months            | Windowed Herzog thresholds                          | 46,381               | 17,894                 | 38.6%               | 7.2%                  |
| healthy ovulatory              | 4 months            | Windowed Herzog with minimum data                   | 37,641               | 13,501                 | 35.9%               | 24.7%                 |
| healthy ovulatory              | 4 months            | Cycle reproducibility, 6-cycle rule                 | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 4 months            | Negative-binomial regression, stabilized dispersion | 37,641               | 1,236                  | 3.3%                | 24.7%                 |
| healthy ovulatory              | 6 cycles            | Exact Herzog 2004, any CE pattern                   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 6 cycles            | Windowed Herzog thresholds                          | 47,394               | 16,299                 | 34.4%               | 5.2%                  |
| healthy ovulatory              | 6 cycles            | Windowed Herzog with minimum data                   | 40,861               | 12,925                 | 31.6%               | 18.3%                 |
| healthy ovulatory              | 6 cycles            | Cycle reproducibility, 6-cycle rule                 | 22,591               | 2,742                  | 12.1%               | 54.8%                 |
| healthy ovulatory              | 6 cycles            | Negative-binomial regression, stabilized dispersion | 40,861               | 1,860                  | 4.6%                | 18.3%                 |
| healthy ovulatory              | 6 months            | Exact Herzog 2004, any CE pattern                   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 6 months            | Windowed Herzog thresholds                          | 47,489               | 16,185                 | 34.1%               | 5.0%                  |
| healthy ovulatory              | 6 months            | Windowed Herzog with minimum data                   | 41,136               | 12,985                 | 31.6%               | 17.7%                 |
| healthy ovulatory              | 6 months            | Cycle reproducibility, 6-cycle rule                 | 6,706                | 704                    | 10.5%               | 86.6%                 |
| healthy ovulatory              | 6 months            | Negative-binomial regression, stabilized dispersion | 41,136               | 1,881                  | 4.6%                | 17.7%                 |
| heterogeneous menstruating-age | 1 month             | Exact Herzog 2004, any CE pattern                   | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 1 month             | Windowed Herzog thresholds                          | 38,083               | 19,748                 | 51.9%               | 23.8%                 |
| heterogeneous menstruating-age | 1 month             | Windowed Herzog with minimum data                   | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 1 month             | Cycle reproducibility, 6-cycle rule                 | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 1 month             | Negative-binomial regression, stabilized dispersion | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 12 cycles           | Exact Herzog 2004, any CE pattern                   | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 12 cycles           | Windowed Herzog thresholds                          | 48,706               | 20,061                 | 41.2%               | 2.6%                  |
| heterogeneous menstruating-age | 12 cycles           | Windowed Herzog with minimum data                   | 45,332               | 18,002                 | 39.7%               | 9.3%                  |
| heterogeneous menstruating-age | 12 cycles           | Cycle reproducibility, 6-cycle rule                 | 20,074               | 1,854                  | 9.2%                | 59.9%                 |
| heterogeneous menstruating-age | 12 cycles           | Negative-binomial regression, stabilized dispersion | 45,332               | 2,153                  | 4.7%                | 9.3%                  |
| heterogeneous menstruating-age | 12 months           | Exact Herzog 2004, any CE pattern                   | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 12 months           | Windowed Herzog thresholds                          | 48,710               | 20,543                 | 42.2%               | 2.6%                  |
| heterogeneous menstruating-age | 12 months           | Windowed Herzog with minimum data                   | 45,303               | 18,437                 | 40.7%               | 9.4%                  |
| heterogeneous menstruating-age | 12 months           | Cycle reproducibility, 6-cycle rule                 | 20,438               | 1,279                  | 6.3%                | 59.1%                 |
| heterogeneous menstruating-age | 12 months           | Negative-binomial regression, stabilized dispersion | 45,303               | 2,122                  | 4.7%                | 9.4%                  |
| heterogeneous menstruating-age | 3 cycles            | Exact Herzog 2004, any CE pattern                   | 16,673               | 8,602                  | 51.6%               | 66.7%                 |
| heterogeneous menstruating-age | 3 cycles            | Windowed Herzog thresholds                          | 45,349               | 22,907                 | 50.5%               | 9.3%                  |
| heterogeneous menstruating-age | 3 cycles            | Windowed Herzog with minimum data                   | 1,947                | 1,140                  | 58.6%               | 96.1%                 |
| heterogeneous menstruating-age | 3 cycles            | Cycle reproducibility, 6-cycle rule                 | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 cycles            | Negative-binomial regression, stabilized dispersion | 1,947                | 50                     | 2.6%                | 96.1%                 |
| heterogeneous menstruating-age | 3 months            | Exact Herzog 2004, any CE pattern                   | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 months            | Windowed Herzog thresholds                          | 45,297               | 23,184                 | 51.2%               | 9.4%                  |
| heterogeneous menstruating-age | 3 months            | Windowed Herzog with minimum data                   | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 months            | Cycle reproducibility, 6-cycle rule                 | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 months            | Negative-binomial regression, stabilized dispersion | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 36-month full diary | Exact Herzog 2004, any CE pattern                   | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 36-month full diary | Windowed Herzog thresholds                          | 49,587               | 17,983                 | 36.3%               | 0.8%                  |
| heterogeneous menstruating-age | 36-month full diary | Windowed Herzog with minimum data                   | 48,333               | 17,206                 | 35.6%               | 3.3%                  |
| heterogeneous menstruating-age | 36-month full diary | Cycle reproducibility, 6-cycle rule                 | 13,364               | 1,605                  | 12.0%               | 73.3%                 |
| heterogeneous menstruating-age | 36-month full diary | Negative-binomial regression, stabilized dispersion | 48,333               | 2,032                  | 4.2%                | 3.3%                  |
| heterogeneous menstruating-age | 4 months            | Exact Herzog 2004, any CE pattern                   | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 4 months            | Windowed Herzog thresholds                          | 46,369               | 23,060                 | 49.7%               | 7.3%                  |
| heterogeneous menstruating-age | 4 months            | Windowed Herzog with minimum data                   | 37,530               | 17,825                 | 47.5%               | 24.9%                 |
| heterogeneous menstruating-age | 4 months            | Cycle reproducibility, 6-cycle rule                 | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 4 months            | Negative-binomial regression, stabilized dispersion | 37,529               | 1,180                  | 3.1%                | 24.9%                 |
| heterogeneous menstruating-age | 6 cycles            | Exact Herzog 2004, any CE pattern                   | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 6 cycles            | Windowed Herzog thresholds                          | 47,488               | 22,375                 | 47.1%               | 5.0%                  |
| heterogeneous menstruating-age | 6 cycles            | Windowed Herzog with minimum data                   | 41,326               | 18,672                 | 45.2%               | 17.3%                 |
| heterogeneous menstruating-age | 6 cycles            | Cycle reproducibility, 6-cycle rule                 | 25,842               | 2,006                  | 7.8%                | 48.3%                 |
| heterogeneous menstruating-age | 6 cycles            | Negative-binomial regression, stabilized dispersion | 41,326               | 1,910                  | 4.6%                | 17.3%                 |
| heterogeneous menstruating-age | 6 months            | Exact Herzog 2004, any CE pattern                   | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 6 months            | Windowed Herzog thresholds                          | 47,490               | 22,546                 | 47.5%               | 5.0%                  |
| heterogeneous menstruating-age | 6 months            | Windowed Herzog with minimum data                   | 41,240               | 18,769                 | 45.5%               | 17.5%                 |
| heterogeneous menstruating-age | 6 months            | Cycle reproducibility, 6-cycle rule                 | 5,189                | 409                    | 7.9%                | 89.6%                 |
| heterogeneous menstruating-age | 6 months            | Negative-binomial regression, stabilized dispersion | 41,240               | 1,862                  | 4.5%                | 17.5%                 |

**Table 3 caption.** False-positive and indeterminate rates for every prespecified observation window and core definition. Exact Herzog 2004 is expected to be classifiable only for 3-complete-cycle windows.

## Table 4. Null n=30 study-level prevalence benchmarks

**Why this table is included.** This table maps person-level false positives into apparent prevalence in small studies. It reports prevalence among all 30 participants, prevalence among classifiable participants only, and the probability of exceeding the 39.1% and 44.2% benchmark values.

**Code to call.**

In [ ]:
study_benchmarks = summary_tables[
    (summary_tables.table_type == "study_level_3month_n30")
    & (summary_tables.definition.isin([
        "A_windowed_any", "B_minimum_data_any",
        "C_reproducibility_any", "D_nb_regression_any"
    ]))
].copy()
study_benchmarks


| Cohort                         | CE definition                                       | Analysis denominator           | Monte Carlo studies | Mean apparent CE prevalence | 2.5th percentile | 97.5th percentile | Probability prevalence at least 39.1% | Probability prevalence at least 44.2% |
| ------------------------------ | --------------------------------------------------- | ------------------------------ | ------------------- | --------------------------- | ---------------- | ----------------- | ------------------------------------- | ------------------------------------- |
| healthy ovulatory              | Windowed Herzog thresholds                          | All 30 participants            | 10,000              | 37.5%                       | 20.0%            | 53.3%             | 45.9%                                 | 19.7%                                 |
| healthy ovulatory              | Windowed Herzog thresholds                          | Classifiable participants only | 10,000              | 41.4%                       | 23.1%            | 60.0%             | 60.8%                                 | 39.5%                                 |
| healthy ovulatory              | Windowed Herzog with minimum data                   | All 30 participants            | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Windowed Herzog with minimum data                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Cycle reproducibility, 6-cycle rule                 | All 30 participants            | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Cycle reproducibility, 6-cycle rule                 | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Negative-binomial regression, stabilized dispersion | All 30 participants            | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Negative-binomial regression, stabilized dispersion | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Windowed Herzog thresholds                          | All 30 participants            | 10,000              | 46.0%                       | 30.0%            | 63.3%             | 79.9%                                 | 54.5%                                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                          | Classifiable participants only | 10,000              | 50.8%                       | 32.0%            | 69.2%             | 89.4%                                 | 75.9%                                 |
| heterogeneous menstruating-age | Windowed Herzog with minimum data                   | All 30 participants            | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Windowed Herzog with minimum data                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Cycle reproducibility, 6-cycle rule                 | All 30 participants            | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Cycle reproducibility, 6-cycle rule                 | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Negative-binomial regression, stabilized dispersion | All 30 participants            | 10,000              | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Negative-binomial regression, stabilized dispersion | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |

**Table 4 caption.** Study-level Monte Carlo summary from 10,000 null studies of 30 participants using 3-month windows. The interval columns are the 2.5th and 97.5th percentiles of study-level apparent prevalence.

## Table 5. Trial-like conditioned subsets

**Why this table is included.** These subsets answer whether common enrollment restrictions reduce false positives or mainly change the classifiable denominator. The common-classifiable subset supports head-to-head comparisons because every listed core definition is defined on the same windows.

**Code to call.**

In [ ]:
trial_like_subsets = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.window_type == "full")
    & (summary_tables.subset.isin([
        "ge_1_seizure_day_per_month",
        "ge_2_seizures_per_month",
        "strict_23_35_day_cycles_only",
        "common_classifiable_subset",
    ]))
    & (summary_tables.definition.isin([
        "A_windowed_any", "B_minimum_data_any",
        "C_reproducibility_any", "D_nb_regression_any"
    ]))
].copy()
trial_like_subsets


| Cohort                         | Analysis denominator             | CE definition                                       | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows |
| ------------------------------ | -------------------------------- | --------------------------------------------------- | -------------------- | ---------------------- | ---------------------------- | --------------------- |
| healthy ovulatory              | Common classifiable subset       | Windowed Herzog thresholds                          | 6,132                | 135                    | 2.2% (1.9, 2.6)              | 0.0%                  |
| healthy ovulatory              | Common classifiable subset       | Windowed Herzog with minimum data                   | 6,132                | 135                    | 2.2% (1.9, 2.6)              | 0.0%                  |
| healthy ovulatory              | Common classifiable subset       | Cycle reproducibility, 6-cycle rule                 | 6,132                | 1                      | 0.0% (0.0, 0.1)              | 0.0%                  |
| healthy ovulatory              | Common classifiable subset       | Negative-binomial regression, stabilized dispersion | 6,132                | 156                    | 2.5% (2.2, 3.0)              | 0.0%                  |
| healthy ovulatory              | At least 1 seizure day per month | Windowed Herzog thresholds                          | 37,309               | 1,952                  | 5.2% (5.0, 5.5)              | 0.0%                  |
| healthy ovulatory              | At least 1 seizure day per month | Windowed Herzog with minimum data                   | 37,309               | 1,952                  | 5.2% (5.0, 5.5)              | 0.0%                  |
| healthy ovulatory              | At least 1 seizure day per month | Cycle reproducibility, 6-cycle rule                 | 6,132                | 1                      | 0.0% (0.0, 0.1)              | 83.6%                 |
| healthy ovulatory              | At least 1 seizure day per month | Negative-binomial regression, stabilized dispersion | 37,309               | 1,614                  | 4.3% (4.1, 4.5)              | 0.0%                  |
| healthy ovulatory              | At least 2 seizures per month    | Windowed Herzog thresholds                          | 30,106               | 1,179                  | 3.9% (3.7, 4.1)              | 0.0%                  |
| healthy ovulatory              | At least 2 seizures per month    | Windowed Herzog with minimum data                   | 30,106               | 1,179                  | 3.9% (3.7, 4.1)              | 0.0%                  |
| healthy ovulatory              | At least 2 seizures per month    | Cycle reproducibility, 6-cycle rule                 | 6,132                | 1                      | 0.0% (0.0, 0.1)              | 79.6%                 |
| healthy ovulatory              | At least 2 seizures per month    | Negative-binomial regression, stabilized dispersion | 30,106               | 1,299                  | 4.3% (4.1, 4.6)              | 0.0%                  |
| healthy ovulatory              | Strict 23-35 day cycles only     | Windowed Herzog thresholds                          | 5,407                | 580                    | 10.7% (9.9, 11.6)            | 0.9%                  |
| healthy ovulatory              | Strict 23-35 day cycles only     | Windowed Herzog with minimum data                   | 5,275                | 502                    | 9.5% (8.8, 10.3)             | 3.3%                  |
| healthy ovulatory              | Strict 23-35 day cycles only     | Cycle reproducibility, 6-cycle rule                 | 760                  | 0                      | 0.0% (0.0, 0.5)              | 86.1%                 |
| healthy ovulatory              | Strict 23-35 day cycles only     | Negative-binomial regression, stabilized dispersion | 5,275                | 204                    | 3.9% (3.4, 4.4)              | 3.3%                  |
| heterogeneous menstruating-age | Common classifiable subset       | Windowed Herzog thresholds                          | 13,364               | 3,634                  | 27.2% (26.4, 28.0)           | 0.0%                  |
| heterogeneous menstruating-age | Common classifiable subset       | Windowed Herzog with minimum data                   | 13,364               | 3,634                  | 27.2% (26.4, 28.0)           | 0.0%                  |
| heterogeneous menstruating-age | Common classifiable subset       | Cycle reproducibility, 6-cycle rule                 | 13,364               | 1,605                  | 12.0% (11.5, 12.6)           | 0.0%                  |
| heterogeneous menstruating-age | Common classifiable subset       | Negative-binomial regression, stabilized dispersion | 13,364               | 565                    | 4.2% (3.9, 4.6)              | 0.0%                  |
| heterogeneous menstruating-age | At least 1 seizure day per month | Windowed Herzog thresholds                          | 37,196               | 11,764                 | 31.6% (31.2, 32.1)           | 0.0%                  |
| heterogeneous menstruating-age | At least 1 seizure day per month | Windowed Herzog with minimum data                   | 37,196               | 11,764                 | 31.6% (31.2, 32.1)           | 0.0%                  |
| heterogeneous menstruating-age | At least 1 seizure day per month | Cycle reproducibility, 6-cycle rule                 | 13,273               | 1,526                  | 11.5% (11.0, 12.1)           | 64.3%                 |
| heterogeneous menstruating-age | At least 1 seizure day per month | Negative-binomial regression, stabilized dispersion | 37,196               | 1,633                  | 4.4% (4.2, 4.6)              | 0.0%                  |
| heterogeneous menstruating-age | At least 2 seizures per month    | Windowed Herzog thresholds                          | 30,016               | 9,006                  | 30.0% (29.5, 30.5)           | 0.0%                  |
| heterogeneous menstruating-age | At least 2 seizures per month    | Windowed Herzog with minimum data                   | 30,016               | 9,006                  | 30.0% (29.5, 30.5)           | 0.0%                  |
| heterogeneous menstruating-age | At least 2 seizures per month    | Cycle reproducibility, 6-cycle rule                 | 12,811               | 1,197                  | 9.3% (8.9, 9.9)              | 57.3%                 |
| heterogeneous menstruating-age | At least 2 seizures per month    | Negative-binomial regression, stabilized dispersion | 30,016               | 1,292                  | 4.3% (4.1, 4.5)              | 0.0%                  |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Windowed Herzog thresholds                          | 2,950                | 1,158                  | 39.3% (37.5, 41.0)           | 1.1%                  |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Windowed Herzog with minimum data                   | 2,878                | 1,116                  | 38.8% (37.0, 40.6)           | 3.5%                  |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Cycle reproducibility, 6-cycle rule                 | 454                  | 3                      | 0.7% (0.2, 1.9)              | 84.8%                 |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Negative-binomial regression, stabilized dispersion | 2,878                | 115                    | 4.0% (3.3, 4.8)              | 3.5%                  |

**Table 5 caption.** Full-diary false-positive rates after applying trial-like eligibility restrictions or a common classifiable denominator. This separates changes in apparent risk from changes in analyzability.

## Table 6. C1/C2/C3 decomposition and C3 exclusion

**Why this table is included.** This table directly addresses whether the heterogeneous-cohort signal is driven by C3 logic. It reports any CE, any CE excluding C3, and pattern-specific C1/C2/C3 rows for full-diary windows.

**Code to call.**

In [ ]:
pattern_decomposition = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.subset == "all")
    & (summary_tables.window_type == "full")
    & (summary_tables.definition.isin([
        "A_windowed_any", "A_windowed_excluding_C3",
        "A_windowed_C1_only", "A_windowed_C2_only", "A_windowed_C3_only",
        "B_minimum_data_any", "B_minimum_data_excluding_C3"
    ]))
].copy()
pattern_decomposition


| Cohort                         | CE definition                                   | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows |
| ------------------------------ | ----------------------------------------------- | -------------------- | ---------------------- | ---------------------------- | --------------------- |
| healthy ovulatory              | Windowed Herzog C1 only                         | 49,451               | 3,893                  | 7.9% (7.6, 8.1)              | 1.1%                  |
| healthy ovulatory              | Windowed Herzog C2 only                         | 49,495               | 2,753                  | 5.6% (5.4, 5.8)              | 1.0%                  |
| healthy ovulatory              | Windowed Herzog C3 only                         | 0                    | 0                      | NA                           | 100.0%                |
| healthy ovulatory              | Windowed Herzog thresholds                      | 49,605               | 5,576                  | 11.2% (11.0, 11.5)           | 0.8%                  |
| healthy ovulatory              | Windowed Herzog thresholds, excluding C3        | 49,605               | 5,576                  | 11.2% (11.0, 11.5)           | 0.8%                  |
| healthy ovulatory              | Windowed Herzog with minimum data               | 48,359               | 4,940                  | 10.2% (9.9, 10.5)            | 3.3%                  |
| healthy ovulatory              | Windowed Herzog with minimum data, excluding C3 | 48,359               | 4,940                  | 10.2% (9.9, 10.5)            | 3.3%                  |
| heterogeneous menstruating-age | Windowed Herzog C1 only                         | 49,392               | 4,171                  | 8.4% (8.2, 8.7)              | 1.2%                  |
| heterogeneous menstruating-age | Windowed Herzog C2 only                         | 49,482               | 2,788                  | 5.6% (5.4, 5.8)              | 1.0%                  |
| heterogeneous menstruating-age | Windowed Herzog C3 only                         | 38,989               | 14,425                 | 37.0% (36.5, 37.5)           | 22.0%                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                      | 49,587               | 17,983                 | 36.3% (35.8, 36.7)           | 0.8%                  |
| heterogeneous menstruating-age | Windowed Herzog thresholds, excluding C3        | 49,587               | 5,837                  | 11.8% (11.5, 12.1)           | 0.8%                  |
| heterogeneous menstruating-age | Windowed Herzog with minimum data               | 48,333               | 17,206                 | 35.6% (35.2, 36.0)           | 3.3%                  |
| heterogeneous menstruating-age | Windowed Herzog with minimum data, excluding C3 | 48,333               | 5,183                  | 10.7% (10.5, 11.0)           | 3.3%                  |

**Table 6 caption.** Full-diary pattern decomposition and C3-exclusion sensitivity. C3 is evaluated only when ILP logic is applicable.

## Table 7. Negative-binomial dispersion sensitivity

**Why this table is included.** This table separates the stabilized-dispersion regression comparator from a non-oracle window-only dispersion sensitivity.

**Code to call.**

In [ ]:
nb_dispersion_sensitivity = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.subset == "all")
    & (summary_tables.window_type == "full")
    & (summary_tables.definition.isin([
        "D_nb_regression_any", "D_nb_regression_window_alpha_any"
    ]))
].copy()
nb_dispersion_sensitivity


| Cohort                         | Observation window  | CE definition                                        | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows |
| ------------------------------ | ------------------- | ---------------------------------------------------- | -------------------- | ---------------------- | ---------------------------- | --------------------- |
| healthy ovulatory              | 36-month full diary | Negative-binomial regression, stabilized dispersion  | 48,359               | 1,966                  | 4.1% (3.9, 4.2)              | 3.3%                  |
| healthy ovulatory              | 36-month full diary | Negative-binomial regression, window-only dispersion | 48,359               | 1,966                  | 4.1% (3.9, 4.2)              | 3.3%                  |
| heterogeneous menstruating-age | 36-month full diary | Negative-binomial regression, stabilized dispersion  | 48,333               | 2,032                  | 4.2% (4.0, 4.4)              | 3.3%                  |
| heterogeneous menstruating-age | 36-month full diary | Negative-binomial regression, window-only dispersion | 48,333               | 2,032                  | 4.2% (4.0, 4.4)              | 3.3%                  |

**Table 7 caption.** Negative-binomial apparent classification rates using full-diary stabilized alpha and window-only alpha. Both use the same M/O model and Holm family.

## Table 8. Seizure-frequency and cycle-regularity strata

**Why this table is included.** The requested strata diagnose where false positives concentrate. Seizure-frequency strata use observed full-diary seizure-days per month; cycle-regularity strata use observed participant-level cycle-length SD.

**Code to call.**

In [ ]:
strata_rows = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.window_type == "full")
    & (summary_tables.definition.isin(["A_windowed_any", "B_minimum_data_any", "D_nb_regression_any"]))
    & (
        summary_tables.subset.astype(str).str.startswith("seizure_frequency:")
        | summary_tables.subset.astype(str).str.startswith("cycle_regularity:")
    )
].copy()
strata_rows


| Cohort                         | Stratum type              | Stratum                           | CE definition                                       | Classifiable windows | False-positive windows | False-positive rate | Indeterminate windows |
| ------------------------------ | ------------------------- | --------------------------------- | --------------------------------------------------- | -------------------- | ---------------------- | ------------------- | --------------------- |
| healthy ovulatory              | Cycle-regularity stratum  | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds                          | 40,112               | 4,510                  | 11.2%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum  | Cycle length SD 2 to <4 days      | Windowed Herzog with minimum data                   | 39,115               | 4,000                  | 10.2%               | 3.2%                  |
| healthy ovulatory              | Cycle-regularity stratum  | Cycle length SD 2 to <4 days      | Negative-binomial regression, stabilized dispersion | 39,115               | 1,592                  | 4.1%                | 3.2%                  |
| healthy ovulatory              | Cycle-regularity stratum  | Cycle length SD < 2 days          | Windowed Herzog thresholds                          | 472                  | 56                     | 11.9%               | 0.8%                  |
| healthy ovulatory              | Cycle-regularity stratum  | Cycle length SD < 2 days          | Windowed Herzog with minimum data                   | 451                  | 43                     | 9.5%                | 5.3%                  |
| healthy ovulatory              | Cycle-regularity stratum  | Cycle length SD < 2 days          | Negative-binomial regression, stabilized dispersion | 451                  | 15                     | 3.3%                | 5.3%                  |
| healthy ovulatory              | Cycle-regularity stratum  | Cycle length SD at least 4 days   | Windowed Herzog thresholds                          | 9,021                | 1,010                  | 11.2%               | 0.9%                  |
| healthy ovulatory              | Cycle-regularity stratum  | Cycle length SD at least 4 days   | Windowed Herzog with minimum data                   | 8,793                | 897                    | 10.2%               | 3.4%                  |
| healthy ovulatory              | Cycle-regularity stratum  | Cycle length SD at least 4 days   | Negative-binomial regression, stabilized dispersion | 8,793                | 359                    | 4.1%                | 3.4%                  |
| healthy ovulatory              | Seizure-frequency stratum | 1 to <4 seizure days per month    | Windowed Herzog thresholds                          | 27,175               | 1,701                  | 6.3%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum | 1 to <4 seizure days per month    | Windowed Herzog with minimum data                   | 27,175               | 1,701                  | 6.3%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum | 1 to <4 seizure days per month    | Negative-binomial regression, stabilized dispersion | 27,175               | 1,267                  | 4.7%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum | 4 to <10 seizure days per month   | Windowed Herzog thresholds                          | 10,134               | 251                    | 2.5%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum | 4 to <10 seizure days per month   | Windowed Herzog with minimum data                   | 10,134               | 251                    | 2.5%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum | 4 to <10 seizure days per month   | Negative-binomial regression, stabilized dispersion | 10,134               | 347                    | 3.4%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum | Less than 1 seizure day per month | Windowed Herzog thresholds                          | 12,296               | 3,624                  | 29.5%               | 3.1%                  |
| healthy ovulatory              | Seizure-frequency stratum | Less than 1 seizure day per month | Windowed Herzog with minimum data                   | 11,050               | 2,988                  | 27.0%               | 12.9%                 |
| healthy ovulatory              | Seizure-frequency stratum | Less than 1 seizure day per month | Negative-binomial regression, stabilized dispersion | 11,050               | 352                    | 3.2%                | 12.9%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum  | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds                          | 24,722               | 9,854                  | 39.9%               | 0.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum  | Cycle length SD 2 to <4 days      | Windowed Herzog with minimum data                   | 24,102               | 9,497                  | 39.4%               | 3.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum  | Cycle length SD 2 to <4 days      | Negative-binomial regression, stabilized dispersion | 24,102               | 1,008                  | 4.2%                | 3.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum  | Cycle length SD < 2 days          | Windowed Herzog thresholds                          | 262                  | 92                     | 35.1%               | 1.9%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum  | Cycle length SD < 2 days          | Windowed Herzog with minimum data                   | 255                  | 87                     | 34.1%               | 4.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum  | Cycle length SD < 2 days          | Negative-binomial regression, stabilized dispersion | 255                  | 11                     | 4.3%                | 4.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum  | Cycle length SD at least 4 days   | Windowed Herzog thresholds                          | 24,603               | 8,037                  | 32.7%               | 0.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum  | Cycle length SD at least 4 days   | Windowed Herzog with minimum data                   | 23,976               | 7,622                  | 31.8%               | 3.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum  | Cycle length SD at least 4 days   | Negative-binomial regression, stabilized dispersion | 23,976               | 1,013                  | 4.2%                | 3.3%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum | 1 to <4 seizure days per month    | Windowed Herzog thresholds                          | 27,072               | 9,113                  | 33.7%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum | 1 to <4 seizure days per month    | Windowed Herzog with minimum data                   | 27,072               | 9,113                  | 33.7%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum | 1 to <4 seizure days per month    | Negative-binomial regression, stabilized dispersion | 27,072               | 1,305                  | 4.8%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum | 4 to <10 seizure days per month   | Windowed Herzog thresholds                          | 10,124               | 2,651                  | 26.2%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum | 4 to <10 seizure days per month   | Windowed Herzog with minimum data                   | 10,124               | 2,651                  | 26.2%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum | 4 to <10 seizure days per month   | Negative-binomial regression, stabilized dispersion | 10,124               | 328                    | 3.2%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum | Less than 1 seizure day per month | Windowed Herzog thresholds                          | 12,391               | 6,219                  | 50.2%               | 3.2%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum | Less than 1 seizure day per month | Windowed Herzog with minimum data                   | 11,137               | 5,442                  | 48.9%               | 13.0%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum | Less than 1 seizure day per month | Negative-binomial regression, stabilized dispersion | 11,137               | 399                    | 3.6%                | 13.0%                 |

**Table 8 caption.** Full-diary apparent classification rates by observed seizure-frequency and cycle-regularity strata. These strata identify where null positives are concentrated.

## Table 9. Assumption-based historical definitions

**Why this table is included.** Historical rules are exploratory operationalizations rather than literal replications, so they are flagged separately. This table keeps them out of the core endpoint table while still showing their null false-positive behavior.

**Code to call.**

In [ ]:
historical_rows = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.subset == "all")
    & (summary_tables.window_type.isin(["calendar", "full"]))
    & (summary_tables.definition.isin([
        "H1_newmark_penry_any", "H1_newmark_penry_66_7_any",
        "H2_duncan1993_any", "H3_herzog1997_twofold_any",
        "H4_reddy2007_any_phase2x_any"
    ]))
].copy()
historical_rows


| Cohort                         | Observation window  | CE definition                        | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows | Assumption-based historical rule |
| ------------------------------ | ------------------- | ------------------------------------ | -------------------- | ---------------------- | ---------------------------- | --------------------- | -------------------------------- |
| healthy ovulatory              | 3 months            | Newmark-Penry two-thirds sensitivity | 45,280               | 1,870                  | 4.1% (4.0, 4.3)              | 9.4%                  | Yes                              |
| healthy ovulatory              | 3 months            | Newmark-Penry perimenstrual rule     | 45,280               | 3,939                  | 8.7% (8.4, 9.0)              | 9.4%                  | Yes                              |
| healthy ovulatory              | 3 months            | Duncan 1993 ten-day rule             | 45,280               | 3,163                  | 7.0% (6.8, 7.2)              | 9.4%                  | Yes                              |
| healthy ovulatory              | 3 months            | Herzog 1997 twofold rule             | 45,280               | 16,862                 | 37.2% (36.8, 37.7)           | 9.4%                  | Yes                              |
| healthy ovulatory              | 3 months            | Reddy 2007 any-phase twofold rule    | 45,280               | 34,681                 | 76.6% (76.2, 77.0)           | 9.4%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Newmark-Penry two-thirds sensitivity | 49,605               | 170                    | 0.3% (0.3, 0.4)              | 0.8%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Newmark-Penry perimenstrual rule     | 49,605               | 427                    | 0.9% (0.8, 0.9)              | 0.8%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Duncan 1993 ten-day rule             | 49,605               | 361                    | 0.7% (0.7, 0.8)              | 0.8%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Herzog 1997 twofold rule             | 49,605               | 3,603                  | 7.3% (7.0, 7.5)              | 0.8%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Reddy 2007 any-phase twofold rule    | 49,605               | 6,599                  | 13.3% (13.0, 13.6)           | 0.8%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Newmark-Penry two-thirds sensitivity | 45,297               | 1,733                  | 3.8% (3.7, 4.0)              | 9.4%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Newmark-Penry perimenstrual rule     | 45,297               | 3,696                  | 8.2% (7.9, 8.4)              | 9.4%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Duncan 1993 ten-day rule             | 45,297               | 3,018                  | 6.7% (6.4, 6.9)              | 9.4%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Herzog 1997 twofold rule             | 45,297               | 21,417                 | 47.3% (46.8, 47.7)           | 9.4%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Reddy 2007 any-phase twofold rule    | 45,297               | 34,750                 | 76.7% (76.3, 77.1)           | 9.4%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Newmark-Penry two-thirds sensitivity | 49,587               | 175                    | 0.4% (0.3, 0.4)              | 0.8%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Newmark-Penry perimenstrual rule     | 49,587               | 378                    | 0.8% (0.7, 0.8)              | 0.8%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Duncan 1993 ten-day rule             | 49,587               | 311                    | 0.6% (0.6, 0.7)              | 0.8%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Herzog 1997 twofold rule             | 49,587               | 14,914                 | 30.1% (29.7, 30.5)           | 0.8%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Reddy 2007 any-phase twofold rule    | 49,587               | 6,854                  | 13.8% (13.5, 14.1)           | 0.8%                  | Yes                              |

**Table 9 caption.** Apparent classification rates for exploratory historical definitions. These rows are deliberately labeled as assumption-based and should not be interpreted as literal historical replications.

## Table 10. Output manifest

**Why this table is included.** The manifest is machine-readable provenance: it lists every analysis artifact, size, checksum, and the assumptions that were not directly derivable from simulator outputs.

**Code to call.**

In [ ]:
manifest_files = pd.DataFrame(manifest["files"])
manifest_files.assign(size_mb=manifest_files["bytes"] / 1_000_000)[["path", "size_mb", "sha256"]]


| Output file                                                                                  | Size, MB | SHA-256 prefix      |
| -------------------------------------------------------------------------------------------- | -------- | ------------------- |
| outputs/.DS_Store                                                                            | 0.008    | 0b9dfdc22018c24c... |
| outputs/audit_daily_sample.parquet                                                           | 5.109    | 2cdc0bf3628a065f... |
| outputs/fig1_false_positive_by_window.pdf                                                    | 0.019    | a5cbca56b2eb6f2a... |
| outputs/fig1_false_positive_by_window.png                                                    | 0.108    | 1bf4e10d767cc7c1... |
| outputs/fig2_study_prevalence_distribution_3month_n30.pdf                                    | 0.017    | 09c45d92c43c3671... |
| outputs/fig2_study_prevalence_distribution_3month_n30.png                                    | 0.077    | 951ff628fd053882... |
| outputs/fig3_indeterminate_vs_fpr_frontier.pdf                                               | 0.023    | 5feab4a65d2abc89... |
| outputs/fig3_indeterminate_vs_fpr_frontier.png                                               | 0.137    | c0a28a1453bc217f... |
| outputs/fig4_historical_vs_core_definitions.pdf                                              | 0.018    | 49206c2dc2206c4b... |
| outputs/fig4_historical_vs_core_definitions.png                                              | 0.068    | 971b30c95b0d1949... |
| outputs/fig5_null_cycle_day_profile.pdf                                                      | 0.017    | 3e9d4c45363d5230... |
| outputs/fig5_null_cycle_day_profile.png                                                      | 0.144    | 020ae0b61c10d8e7... |
| outputs/neurology_submission/fig1_false_positive_by_window.pdf                               | 0.019    | 8fa931203262567c... |
| outputs/neurology_submission/fig1_false_positive_by_window.png                               | 0.105    | f6c288375b203f65... |
| outputs/neurology_submission/fig2_study_prevalence_distribution_3month_n30.pdf               | 0.017    | ef69b89acdb57703... |
| outputs/neurology_submission/fig2_study_prevalence_distribution_3month_n30.png               | 0.078    | 89c0e9fc207e744b... |
| outputs/neurology_submission/fig3_indeterminate_vs_fpr_frontier.pdf                          | 0.023    | 09f4d2c321c179d5... |
| outputs/neurology_submission/fig3_indeterminate_vs_fpr_frontier.png                          | 0.137    | 207f9bac3ccd947a... |
| outputs/neurology_submission/fig4_historical_vs_core_definitions.pdf                         | 0.018    | d048c7fa80c7ecfa... |
| outputs/neurology_submission/fig4_historical_vs_core_definitions.png                         | 0.067    | fe6b3ace3ed5983f... |
| outputs/neurology_submission/fig5_null_cycle_day_profile.pdf                                 | 0.017    | edbe3876eafe8ce2... |
| outputs/neurology_submission/fig5_null_cycle_day_profile.png                                 | 0.143    | 514e9bb0351f8aee... |
| outputs/neurology_submission/paper1_null_ce_appendix_draft.docx                              | 0.369    | 11a7615469e8ad88... |
| outputs/neurology_submission/paper1_null_ce_neurology_original_article_draft.docx            | 0.369    | ae6b63f65a801ced... |
| outputs/neurology_submission/render_appendix/page-1.png                                      | 0.160    | d6b7f4e26d1c585a... |
| outputs/neurology_submission/render_appendix/page-10.png                                     | 0.182    | 55eb67e405599e77... |
| outputs/neurology_submission/render_appendix/page-11.png                                     | 0.217    | 0cb1d7fe27a93e27... |
| outputs/neurology_submission/render_appendix/page-12.png                                     | 0.058    | 46667002a1ba7d98... |
| outputs/neurology_submission/render_appendix/page-13.png                                     | 0.208    | 325888547db061ca... |
| outputs/neurology_submission/render_appendix/page-14.png                                     | 0.115    | 4cfef93959f3bb28... |
| outputs/neurology_submission/render_appendix/page-15.png                                     | 0.290    | 193ecc78e60b7dd6... |
| outputs/neurology_submission/render_appendix/page-16.png                                     | 0.071    | 94dceafa61c9bfe8... |
| outputs/neurology_submission/render_appendix/page-2.png                                      | 0.156    | a1c86f49cc9ec153... |
| outputs/neurology_submission/render_appendix/page-3.png                                      | 0.186    | a1e92b4f7ffa496e... |
| outputs/neurology_submission/render_appendix/page-4.png                                      | 0.188    | 527d02edf01f3d98... |
| outputs/neurology_submission/render_appendix/page-5.png                                      | 0.181    | 66ce9d9e29f4aaf1... |
| outputs/neurology_submission/render_appendix/page-6.png                                      | 0.187    | b6a59d02a77e2d50... |
| outputs/neurology_submission/render_appendix/page-7.png                                      | 0.179    | 3ee7e2259771c5d0... |
| outputs/neurology_submission/render_appendix/page-8.png                                      | 0.177    | a6be8eb7f3a477b7... |
| outputs/neurology_submission/render_appendix/page-9.png                                      | 0.166    | aaeba0f19af33976... |
| outputs/neurology_submission/render_appendix/paper1_null_ce_appendix_draft.pdf               | 0.807    | 78f6eb65a69f05d7... |
| outputs/neurology_submission/render_main/page-1.png                                          | 0.123    | d356b2ac6959ab04... |
| outputs/neurology_submission/render_main/page-10.png                                         | 0.047    | 457218b1592952ee... |
| outputs/neurology_submission/render_main/page-11.png                                         | 0.254    | af44492921f25a4f... |
| outputs/neurology_submission/render_main/page-12.png                                         | 0.034    | 96ff5a57b5b53507... |
| outputs/neurology_submission/render_main/page-13.png                                         | 0.241    | 2208c41d5580eea3... |
| outputs/neurology_submission/render_main/page-14.png                                         | 0.079    | 2a695071b7599742... |
| outputs/neurology_submission/render_main/page-15.png                                         | 0.193    | 109a1ed6fde4c401... |
| outputs/neurology_submission/render_main/page-16.png                                         | 0.244    | 3b0097bc0b4a65a0... |
| outputs/neurology_submission/render_main/page-2.png                                          | 0.264    | c46f1b0171ed0fab... |
| outputs/neurology_submission/render_main/page-3.png                                          | 0.034    | 7c5b7094f55d67c9... |
| outputs/neurology_submission/render_main/page-4.png                                          | 0.235    | 5e5e297d3773cf2a... |
| outputs/neurology_submission/render_main/page-5.png                                          | 0.276    | f941a825c172ae9f... |
| outputs/neurology_submission/render_main/page-6.png                                          | 0.245    | a782dff9e092025c... |
| outputs/neurology_submission/render_main/page-7.png                                          | 0.264    | 0da979a2a6ca040a... |
| outputs/neurology_submission/render_main/page-8.png                                          | 0.257    | ce210ec9518722cb... |
| outputs/neurology_submission/render_main/page-9.png                                          | 0.235    | 004d3b8b474e3643... |
| outputs/neurology_submission/render_main/paper1_null_ce_neurology_original_article_draft.pdf | 0.502    | f894652d61856455... |
| outputs/neurology_submission/second_pass_review_note.txt                                     | 0.001    | 5b433cef20e30c0e... |
| outputs/neurology_submission/~$per1_null_ce_neurology_original_article_draft.docx            | 0.000    | 2e65f4ee45070a62... |
| outputs/participant_summary.parquet                                                          | 7.031    | 4c8b6c96ee70ddf1... |
| outputs/progress.json                                                                        | 0.000    | 212e112e5f238789... |
| outputs/study_level_3month_n30.parquet                                                       | 0.794    | 8474520b10a9dc07... |
| outputs/summary_tables.csv                                                                   | 0.657    | 204d8472c01e9acb... |
| outputs/window_results.parquet                                                               | 68.397   | fec1d7e03e7ce616... |

**Table 10 caption.** Machine-readable output provenance. The checksum prefix is included to support reproducibility checks without making the table unnecessarily wide.

## Publication-ready figures

Each figure is written as both PNG for notebook viewing and PDF for publication workflows. Fractional outcomes are displayed on a 0-100% percentage scale. The code cell below is the function call that regenerates the figure set from the populated output tables.

In [ ]:
from paper1_null_ce.core.plots import write_all_figures

# Regenerate PNG and PDF figures from current populated output tables.
write_all_figures(OUTPUT_DIR, summary_tables, study_level, pd.read_parquet(OUTPUT_DIR / "audit_daily_sample.parquet"))


### Figure 1. False-positive rate by window

Primary person-window false-positive rate for core definitions, split by cohort and window length.

PDF companion: [PDF version](../outputs/fig1_false_positive_by_window.pdf)

![Figure 1. False-positive rate by window](../outputs/fig1_false_positive_by_window.png)

**Figure caption.** Apparent classification rates are shown as percentages among classifiable participant windows for each defined observation window. The two cohorts are intentionally displayed separately because they represent different menstrual-cycle assumptions under the null.

### Figure 2. Null n=30 study prevalence distribution

Study-level Monte Carlo distribution for 3-month n=30 null studies, with 39.1% and 44.2% benchmarks.

PDF companion: [PDF version](../outputs/fig2_study_prevalence_distribution_3month_n30.pdf)

![Figure 2. Null n=30 study prevalence distribution](../outputs/fig2_study_prevalence_distribution_3month_n30.png)

**Figure caption.** Each curve summarizes 10,000 simulated studies of 30 participants using random 3-month windows and the windowed Herzog threshold definition. Vertical reference lines mark the benchmark apparent CE prevalence values.

### Figure 3. Indeterminate versus false-positive frontier

Tradeoff between rejecting underspecified windows and the false-positive rate among classifiable windows.

PDF companion: [PDF version](../outputs/fig3_indeterminate_vs_fpr_frontier.pdf)

![Figure 3. Indeterminate versus false-positive frontier](../outputs/fig3_indeterminate_vs_fpr_frontier.png)

**Figure caption.** Each point is a definition-by-window-by-cohort result. Points farther right have more indeterminate windows; points higher on the plot have more false positives among windows that remained classifiable.

### Figure 4. Historical versus core definitions

Assumption-based historical rules compared with core protocol definitions.

PDF companion: [PDF version](../outputs/fig4_historical_vs_core_definitions.pdf)

![Figure 4. Historical versus core definitions](../outputs/fig4_historical_vs_core_definitions.png)

**Figure caption.** The historical definitions are exploratory operationalizations and are plotted next to the core definitions only to show their null false-positive behavior under the same 3-month window setting.

### Figure 5. Null cycle-day seizure profile

Cycle-day seizure profile in the daily audit sample; under the null this should not show true coupling.

PDF companion: [PDF version](../outputs/fig5_null_cycle_day_profile.pdf)

![Figure 5. Null cycle-day seizure profile](../outputs/fig5_null_cycle_day_profile.png)

**Figure caption.** The audit sample contains 1% of participant daily rows. Lines show average daily seizure frequency by observed menstrual cycle day after independent seizure and menstrual diaries were merged under the null.

## Interpretation notes

- The notebook is populated from the current files in `outputs/`. If those files were produced by smoke mode, the numerical values are smoke-test values, not the final 100,000-participant estimates.
- Full-study values are produced by running `run_paper1_null_ce.py --config config.yaml --full`, then rebuilding this notebook.
- Exact Herzog 2004 results are intentionally present only for 3-complete-cycle windows.
- Historical definitions are assumption-based operationalizations and should be kept separate from core endpoints.
- The manifest assumptions are part of the analysis record:

  - Definition D uses a participant-full-diary method-of-moments negative-binomial alpha recorded in d_alpha; Poisson robust fallback is recorded in d_reason when statsmodels NB fitting fails. Definition D_window_alpha re-estimates alpha from the analyzed window as a non-oracle sensitivity.
  - Healthy ovulatory cohort used hormone_cycler build_patient_profile/render_cycle with ovulation_probability set to 1.0 because simulate_diary does not expose a public force-ovulation knob.
  - Historical definitions H1-H4 are assumption-based operationalizations and are flagged in summary outputs.
  - Study-level Monte Carlo samples each selected participant from a deterministic pool of precomputed random valid 3-month windows to avoid retaining all daily diaries in memory.
  - The hormone simulator exposes medical-factor knobs but no natural prevalence sampler; heterogeneous menstruating-age medical factors were sampled from config.yaml rates.